## TurboQuant runbook

**What:** Staged checkpoint-only runs on **`turboquant-experiment`** (QAT int4 TurboQuant MSE-surrogate quant probe)—same staged **`progress.csv`** idea as the Gimlet book (sweep → gate → seeds → signal / table / mini val). **Siblings in this repo:** `colab_int6_gps_int4_mlp_train.ipynb`, `colab_bayesian_backoff_cache_tt_adapter_launcher.ipynb`.

**Paths & run:** **`/content/tqprobe/results/staged`** · Drive **`tqprobe_staged`** · GPU · Run All · Stage B if **`PROCEED_STAGE_B: YES`** · clean slate: **`export FRESH_PROGRESS_CSV=1`** then cell 1.


---
## Code cell 1: Setup
**Drive** mounts in this **Python** cell (not inside a bash subprocess). Then bash clones, deps, tokenizer shard, `CHECKPOINT_ONLY` patch, `run_one.sh`, `progress.csv`.


In [ ]:
# Code cell 1/7 — setup (repos, patch, run_one.sh, progress.csv)
# Drive must run in the notebook kernel. Calling drive.mount from a bash-subprocess
# Python fails on Colab ('NoneType' object has no attribute 'kernel').
import subprocess

from pathlib import Path as _Path

try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    _mp = _Path("/content/drive/MyDrive")
    if _mp.is_dir():
        print("Drive: already mounted at /content/drive (skipping drive.mount).")
    else:
        try:
            drive.mount("/content/drive", force_remount=False)
        except Exception as e:
            print("Drive mount skipped or failed:", e)

bash_script = r'''
# Code cell 1/7 — setup (repos, patch, run_one.sh, progress.csv)
set -eo pipefail
FRESH_PROGRESS_CSV="${FRESH_PROGRESS_CSV:-0}"

# Drive mounts in the Python cell above.

BACKUP_DIR="/content/drive/MyDrive/tqprobe_staged"
if [ -d "/content/drive/MyDrive" ]; then mkdir -p "$BACKUP_DIR"; fi

# --- repos ---
[ -f /content/openai-pg/data/cached_challenge_fineweb.py ] || { rm -rf /content/openai-pg; git clone https://github.com/openai/parameter-golf.git /content/openai-pg; }
[ -f /content/pg/train_gpt.py ] || { rm -rf /content/pg; git clone https://github.com/jmoncayo-pursuit/parameter-golf-qat-int4.git /content/pg; }

if git -C /content/pg fetch origin turboquant-experiment; then
  git -C /content/pg checkout turboquant-experiment 2>/dev/null || true
  git -C /content/pg reset --hard origin/turboquant-experiment
else
  echo "WARN: git fetch failed; keeping existing /content/pg"
fi

mkdir -p /content/pg/logs /content/tqprobe/results/staged
PROG=/content/tqprobe/results/staged/progress.csv
if [ "$FRESH_PROGRESS_CSV" = "1" ]; then
  rm -f "$PROG"
  if [ -d "/content/drive/MyDrive" ]; then rm -f "$BACKUP_DIR/progress.csv"; fi
  echo "RUNBOOK: FRESH_PROGRESS_CSV=1 cleared progress.csv"
fi

# --- deps ---
cd /content/pg
python3 -m pip install -U pip -q || true
pip install -q -r requirements.txt zstandard sentencepiece || pip install -q -r requirements.txt zstandard sentencepiece || true

# --- tokenizer + 1 shard ---
python3 /content/openai-pg/data/cached_challenge_fineweb.py --variant sp1024 --train-shards 1 || \
  { sleep 3; python3 /content/openai-pg/data/cached_challenge_fineweb.py --variant sp1024 --train-shards 1 || true; }

python3 - <<'PY'
from pathlib import Path
p = Path('/content/pg/train_gpt.py')
s = p.read_text()
old_plain = (
    "    enable_cudnn_sdp(False)\n"
    "    enable_flash_sdp(True)\n"
    "    enable_mem_efficient_sdp(False)\n"
    "    enable_math_sdp(False)\n"
    "\n"
    "    logfile = None\n"
)
new_sdpa = (
    "    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:\n"
    "        enable_cudnn_sdp(False)\n"
    "        enable_flash_sdp(True)\n"
    "        enable_mem_efficient_sdp(False)\n"
    "        enable_math_sdp(False)\n"
    "    else:\n"
    "        enable_cudnn_sdp(False)\n"
    "        enable_flash_sdp(False)\n"
    "        enable_mem_efficient_sdp(True)\n"
    "        enable_math_sdp(True)\n"
    "\n"
    "    logfile = None\n"
)
old_guard = (
    "    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:\n"
    "        enable_cudnn_sdp(False)\n"
    "        enable_flash_sdp(True)\n"
    "        enable_mem_efficient_sdp(False)\n"
    "        enable_math_sdp(False)\n"
    "\n"
    "    logfile = None\n"
)
if new_sdpa in s:
    print('SDPA OK (already)')
elif old_plain in s:
    s = s.replace(old_plain, new_sdpa, 1)
    p.write_text(s)
    print('SDPA patch OK')
elif old_guard in s:
    s = s.replace(old_guard, new_sdpa, 1)
    p.write_text(s)
    print('SDPA patch OK')
else:
    print('RUNBOOK_WARN: SDPA block not found')
PY

python3 - <<'PY'
from pathlib import Path
p = Path("/content/pg/train_gpt.py")
s = p.read_text()
anchor = '    args = Hyperparameters()\n'
insert = '    args = Hyperparameters()\n    checkpoint_only = os.environ.get("CHECKPOINT_ONLY", "0") == "1"\n'
if 'checkpoint_only = os.environ.get("CHECKPOINT_ONLY", "0") == "1"' not in s:
    if anchor not in s:
        print('RUNBOOK_WARN: checkpoint_only anchor missing')
    else:
        s = s.replace(anchor, insert, 1)
old = '        should_validate = last_step or (args.val_loss_every > 0 and step % args.val_loss_every == 0)\n'
new = '        should_validate = ((not checkpoint_only) and last_step) or (args.val_loss_every > 0 and step % args.val_loss_every == 0)\n'
co_ok = 'checkpoint_only = os.environ.get("CHECKPOINT_ONLY", "0") == "1"' in s
if new not in s:
    if old not in s:
        print('RUNBOOK_WARN: should_validate line missing')
    elif not co_ok:
        print('RUNBOOK_WARN: skip should_validate')
    else:
        s = s.replace(old, new, 1)
gate = '    quant_obj, quant_stats = quantize_state_dict_int8(base_model.state_dict(), clip_val_map=clip_val_map)'
if 'checkpoint_only: exiting after final_model.pt' not in s:
    if gate not in s:
        old_sv = '        torch.save(base_model.state_dict(), "final_model.pt")\n'
        new_sv = (
            '        torch.save(base_model.state_dict(), "final_model.pt")\n'
            '        if checkpoint_only:\n'
            '            log0("checkpoint_only: exiting after final_model.pt")\n'
            '            return\n'
        )
        if old_sv in s and co_ok:
            s = s.replace(old_sv, new_sv, 1)
        else:
            print('RUNBOOK_WARN: no quantize gate or torch.save patch path')
    elif not co_ok:
        print('RUNBOOK_WARN: skip quantize gate')
    else:
        s = s.replace(
            gate,
            '    if checkpoint_only:\n'
            '        log0("checkpoint_only: exiting after final_model.pt")\n'
            '        return\n\n'
            + gate,
            1,
        )
p.write_text(s)
print('checkpoint_only patch step finished')
PY

# --- runner script ---
mkdir -p /content/pg/logs /content/tqprobe/results/staged

cat >/content/tqprobe/results/staged/run_one.sh <<'EOF_RUNNER'
#!/usr/bin/env bash
set -u
set -o pipefail

SEED="$1"
ITER="$2"
TAG="s${SEED}_i${ITER}"
PG_DIR="/content/pg"
OUT_DIR="/content/tqprobe/results/staged"
BACKUP_DIR="${BACKUP_DIR:-/content/drive/MyDrive/tqprobe_staged}"

backup() {
  if [ -d "/content/drive/MyDrive" ]; then
    mkdir -p "${BACKUP_DIR}"
    cp -f "$1" "${BACKUP_DIR}/" 2>/dev/null || true
  fi
}

echo "==================== ${TAG} TRAIN ===================="
cd "${PG_DIR}"
rm -f final_model.pt

PYTHONUNBUFFERED=1 TORCHDYNAMO_DISABLE=1 CHECKPOINT_ONLY=1 \
DATA_PATH=/content/openai-pg/data/datasets/fineweb10B_sp1024 \
TOKENIZER_PATH=/content/openai-pg/data/tokenizers/fineweb_1024_bpe.model \
VOCAB_SIZE=1024 \
ITERATIONS=${ITER} WARMUP_STEPS=0 VAL_LOSS_EVERY=0 TRAIN_LOG_EVERY=10 EVAL_STRIDE=0 EVAL_CACHE=0 \
TRAIN_BATCH_TOKENS=16384 TRAIN_SEQ_LEN=512 VAL_BATCH_SIZE=65536 \
MAX_WALLCLOCK_SECONDS=7200 COMPRESSOR=lzma SEED=${SEED} \
python3 train_gpt.py > "${PG_DIR}/logs/train_${TAG}.txt" 2>&1
train_rc=$?

backup "${PG_DIR}/logs/train_${TAG}.txt"

if [ $train_rc -ne 0 ] || [ ! -f "${PG_DIR}/final_model.pt" ]; then
  echo "${TAG},FAIL_TRAIN,,,,,,," >> "${OUT_DIR}/progress.csv"
  backup "${OUT_DIR}/progress.csv"
  echo "FAIL ${TAG} train rc=${train_rc}"
  exit 0
fi

cp "${PG_DIR}/final_model.pt" "${OUT_DIR}/final_model_${TAG}.pt"
backup "${OUT_DIR}/final_model_${TAG}.pt"

metrics=$(python3 - "${PG_DIR}/logs/train_${TAG}.txt" "${OUT_DIR}/final_model_${TAG}.pt" <<'PYSUM'
import re, sys, pathlib
log_path = pathlib.Path(sys.argv[1])
ckpt_path = pathlib.Path(sys.argv[2])
lines = log_path.read_text(errors='ignore').splitlines()
pat = re.compile(r"step:(\d+)/(\d+)\s+train_loss:([0-9.]+).*?step_avg:([0-9.]+)ms")
loss = step_avg = ''
for ln in lines:
    m = pat.search(ln)
    if m:
        loss, step_avg = m.group(3), m.group(4)
ckpt_bytes = str(ckpt_path.stat().st_size if ckpt_path.exists() else 0)
print(f"{loss}|{step_avg}|{ckpt_bytes}")
PYSUM
)
IFS='|' read -r last_loss step_avg_ms ckpt_bytes <<< "$metrics"
last_step="${ITER}"
total_steps="${ITER}"

echo "${TAG},OK,${last_step},${total_steps},${last_loss},${step_avg_ms},${ckpt_bytes}," >> "${OUT_DIR}/progress.csv"
backup "${OUT_DIR}/progress.csv"
echo "OK ${TAG} step=${last_step}/${total_steps} loss=${last_loss} step_avg_ms=${step_avg_ms} ckpt_bytes=${ckpt_bytes}"
EOF_RUNNER

chmod +x /content/tqprobe/results/staged/run_one.sh

# progress: restore from Drive if present, else init
PROG=/content/tqprobe/results/staged/progress.csv
if [ "$FRESH_PROGRESS_CSV" != "1" ] && [ -f "$BACKUP_DIR/progress.csv" ] && [ ! -f "$PROG" ]; then
  cp -f "$BACKUP_DIR/progress.csv" "$PROG"
fi
if [ ! -f "$PROG" ]; then
  echo "tag,status,last_step,total_steps,last_loss,step_avg_ms,ckpt_bytes,notes" > "$PROG"
fi

# dedupe; migrate legacy rows (probe columns) → 8-col schema
python3 - <<'PY'
import csv
from io import StringIO
from pathlib import Path

fields = ["tag", "status", "last_step", "total_steps", "last_loss", "step_avg_ms", "ckpt_bytes", "notes"]
p = Path("/content/tqprobe/results/staged/progress.csv")
try:
    raw = p.read_text(encoding="utf-8", errors="replace").splitlines()
except OSError as e:
    print("dedupe: could not read progress.csv:", e)
    raise SystemExit(0)
if len(raw) < 2:
    raise SystemExit(0)
try:
    rows = list(csv.DictReader(StringIO("\n".join(raw))))
except Exception as e:
    print("dedupe: parse error:", e)
    rows = []
if not rows:
    raise SystemExit(0)
norm = []
for r in rows:
    norm.append({k: (r.get(k) or "") for k in fields})

def rank(r):
    ok = 1 if r.get("status") == "OK" else 0
    complete = 1 if (r.get("last_step") or "").strip() and (r.get("last_step") == r.get("total_steps")) else 0
    has_loss = 1 if (r.get("last_loss") or "").strip() else 0
    return (ok, complete, has_loss)

best = {}
for r in norm:
    t = (r.get("tag") or "").strip()
    if not t:
        continue
    if t not in best or rank(r) > rank(best[t]):
        best[t] = r

out = [best[k] for k in sorted(best.keys())]
with p.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(out)
print("deduped progress.csv rows:", len(out))
PY

cp -f "$PROG" "$BACKUP_DIR/progress.csv" 2>/dev/null || true
echo "CELL1_OK backup=$BACKUP_DIR"
cat "$PROG"

'''

_ = subprocess.run(['bash', '-lc', bash_script], check=True)


---
## Code cell 2: STAGE_A — depth sweep
**Goal:** Run `s42_i100` / `s42_i200` / `s42_i300` via `run_one.sh` with 120s heartbeats; skip tags already `OK`.
**Estimated time:** ~20–25 min on a T4-class GPU (observed ~23 min wall for three jobs).


In [ ]:
# Code cell 2/7 — STAGE_A depth sweep
import csv, os, subprocess, time, pathlib, re
from datetime import datetime

import torch

RUNNER = "/content/tqprobe/results/staged/run_one.sh"
PROGRESS = pathlib.Path("/content/tqprobe/results/staged/progress.csv")
LOG_DIR = pathlib.Path("/content/pg/logs")

def rows():
    if not PROGRESS.exists():
        return []
    try:
        with PROGRESS.open(encoding="utf-8", newline="") as f:
            return list(csv.DictReader(f))
    except Exception as e:
        print("⚠️ progress.csv read failed:", e)
        return []

def has_ok(tag):
    return any(r.get("tag") == tag and r.get("status") == "OK" for r in rows())

def last_signal(log_path):
    if not log_path.exists():
        return "(log not created yet)"
    lines = log_path.read_text(errors="ignore").splitlines()
    pat = re.compile(r"(step:|swa:start|checkpoint_only:|RuntimeError|Traceback)")
    for ln in reversed(lines):
        if pat.search(ln):
            return ln
    return "(log exists, no signal line yet)"

def run_tag(seed, iters):
    tag = f"s{seed}_i{iters}"
    log_path = LOG_DIR / f"train_{tag}.txt"
    print(f"\n=== START {tag} @ {datetime.now().isoformat(timespec='seconds')} ===", flush=True)
    p = subprocess.Popen([RUNNER, str(seed), str(iters)])
    t0 = time.time()
    last_beat = t0
    while True:
        rc = p.poll()
        now = time.time()
        if rc is not None:
            print(f"🟢 done {tag} rc={rc} elapsed={int(now-t0)}s", flush=True)
            break
        if now - last_beat >= 120:
            print(f"🔵 heartbeat {tag} @ {datetime.now().isoformat(timespec='seconds')} elapsed={int(now-t0)}s", flush=True)
            print(f"   last_log: {last_signal(log_path)}", flush=True)
            rr = rows()
            if rr:
                t = rr[-1]
                print(
                    f"   progress_tail: {t.get('tag')},{t.get('status')},{t.get('last_loss')},{t.get('ckpt_bytes')}",
                    flush=True,
                )
            last_beat = now
        time.sleep(2)
    print("--- progress so far ---", flush=True)
    try:
        print(PROGRESS.read_text(errors="ignore"), flush=True)
    except OSError as e:
        print("(could not read progress.csv)", e, flush=True)

if not os.path.isfile(RUNNER) or not os.access(RUNNER, os.X_OK):
    print("⏭️ SKIP Stage A: run cell 1 first (missing or non-executable run_one.sh).")
elif not torch.cuda.is_available():
    print("⏭️ SKIP Stage A: no CUDA (Runtime → Change runtime type → GPU).")
else:
    for it in [100, 200, 300]:
        tag = f"s42_i{it}"
        if has_ok(tag):
            print(f"⏭️ skip {tag} (already OK)", flush=True)
        else:
            run_tag(42, it)

---
## Code cell 3: STAGE_A decision (gate)
**Goal:** Require three OK rows, full steps, non-worsening train loss (100→200→300); write `PROCEED_STAGE_B` and `proceed_stage_b.txt` (and Drive backup).
**Estimated time:** typically under 1 min.


In [ ]:
# Code cell 3/7 — STAGE_A_DECISION (gate)
import csv
import shutil
from pathlib import Path


def _safe_int(x, default=0):
    try:
        return int(x)
    except (TypeError, ValueError):
        return default


def _safe_float(x, default=float("nan")):
    try:
        return float(x)
    except (TypeError, ValueError):
        return default


PROGRESS = Path("/content/tqprobe/results/staged/progress.csv")
if not PROGRESS.exists():
    rows = []
    print("⚠️ no progress.csv yet (run cell 1).")
else:
    try:
        with PROGRESS.open(encoding="utf-8", newline="") as f:
            rows = list(csv.DictReader(f))
    except Exception as e:
        print("⚠️ could not read progress.csv:", e)
        rows = []

want = {"s42_i100", "s42_i200", "s42_i300"}
stage_a = [r for r in rows if r.get("tag") in want]
ok = [r for r in stage_a if r.get("status") == "OK"]

loss_by_tag = {}
for r in ok:
    if r.get("last_loss"):
        v = _safe_float(r["last_loss"])
        if v == v:
            loss_by_tag[r["tag"]] = v

complete = (
    len(ok) == 3
    and all(
        r.get("last_step") and r.get("total_steps") and r.get("tag")
        and _safe_int(r["last_step"]) == _safe_int(r["total_steps"]) == _safe_int((r["tag"].split("_i") or ["", "0"])[-1])
        for r in ok
    )
)
trend_ok = (
    all(t in loss_by_tag for t in ["s42_i100", "s42_i200", "s42_i300"])
    and loss_by_tag["s42_i300"] <= loss_by_tag["s42_i200"] <= loss_by_tag["s42_i100"]
)
proceed = len(ok) == 3 and complete and trend_ok

print("Stage A:")
for r in stage_a:
    print(r)
print("\ncomplete:", complete, "trend_ok:", trend_ok)
print("PROCEED_STAGE_B:", "YES" if proceed else "NO")

Path("/content/tqprobe/results/staged/proceed_stage_b.txt").write_text("YES\n" if proceed else "NO\n")

bd = Path("/content/drive/MyDrive/tqprobe_staged")
if bd.parent.exists():
    try:
        bd.mkdir(parents=True, exist_ok=True)
        if PROGRESS.exists():
            shutil.copy(PROGRESS, bd / "progress.csv")
        (bd / "proceed_stage_b.txt").write_text("YES\n" if proceed else "NO\n")
    except OSError as e:
        print("⚠️ Drive backup skipped:", e)

---
## Code cell 4: STAGE_B — cross-seed
**Goal:** If gate is `YES`, run cross-seed grid (`s123`/`s999` × `i200`/`i300`) with the same runner + heartbeats.
**Estimated time:** ~35–40 min on a T4-class GPU (observed ~37 min wall for four jobs).


In [ ]:
# Code cell 4/7 — STAGE_B cross-seed
import csv, os, subprocess, time, pathlib, re, shutil
from datetime import datetime

import torch

FLAG = pathlib.Path("/content/tqprobe/results/staged/proceed_stage_b.txt")
if not FLAG.exists() or FLAG.read_text().strip() != "YES":
    print("Stage B skipped (PROCEED_STAGE_B is not YES).")
else:
    RUNNER = "/content/tqprobe/results/staged/run_one.sh"
    PROGRESS = pathlib.Path("/content/tqprobe/results/staged/progress.csv")
    LOG_DIR = pathlib.Path("/content/pg/logs")

    def rows():
        if not PROGRESS.exists():
            return []
        try:
            with PROGRESS.open(encoding="utf-8", newline="") as f:
                return list(csv.DictReader(f))
        except Exception as e:
            print("⚠️ progress.csv read failed:", e)
            return []

    def has_ok(tag):
        return any(r.get("tag") == tag and r.get("status") == "OK" for r in rows())

    def last_signal(log_path):
        if not log_path.exists():
            return "(log not created yet)"
        lines = log_path.read_text(errors="ignore").splitlines()
        pat = re.compile(r"(step:|swa:start|checkpoint_only:|RuntimeError|Traceback)")
        for ln in reversed(lines):
            if pat.search(ln):
                return ln
        return "(log exists, no signal line yet)"

    def run_tag(seed, iters):
        tag = f"s{seed}_i{iters}"
        log_path = LOG_DIR / f"train_{tag}.txt"
        print(f"\n=== START {tag} @ {datetime.now().isoformat(timespec='seconds')} ===", flush=True)
        p = subprocess.Popen([RUNNER, str(seed), str(iters)])
        t0 = time.time()
        last_beat = t0
        while True:
            rc = p.poll()
            now = time.time()
            if rc is not None:
                print(f"🟢 done {tag} rc={rc} elapsed={int(now-t0)}s", flush=True)
                break
            if now - last_beat >= 120:
                print(f"🔵 heartbeat {tag} @ {datetime.now().isoformat(timespec='seconds')} elapsed={int(now-t0)}s", flush=True)
                print(f"   last_log: {last_signal(log_path)}", flush=True)
                rr = rows()
                if rr:
                    t = rr[-1]
                    print(
                        f"   progress_tail: {t.get('tag')},{t.get('status')},{t.get('last_loss')},{t.get('ckpt_bytes')}",
                        flush=True,
                    )
                last_beat = now
            time.sleep(2)
        print("--- progress so far ---", flush=True)
        try:
            print(PROGRESS.read_text(errors="ignore"), flush=True)
        except OSError as e:
            print("(could not read progress.csv)", e, flush=True)

    if not os.path.isfile(RUNNER) or not os.access(RUNNER, os.X_OK):
        print("⏭️ SKIP Stage B: run cell 1 first (missing or non-executable run_one.sh).")
    elif not torch.cuda.is_available():
        print("⏭️ SKIP Stage B: no CUDA.")
    else:
        for sd in [123, 999]:
            for it in [200, 300]:
                tag = f"s{sd}_i{it}"
                if has_ok(tag):
                    print(f"⏭️ skip {tag} (already OK)", flush=True)
                else:
                    run_tag(sd, it)

    bd = pathlib.Path("/content/drive/MyDrive/tqprobe_staged")
    if bd.parent.exists() and PROGRESS.exists():
        try:
            shutil.copy(PROGRESS, bd / "progress.csv")
        except OSError as e:
            print("⚠️ Drive backup skipped:", e)

---
## Code cell 5: BASELINE_CHECKPOINT_SIGNAL_STRONG
**Goal:** Aggregate checkpoint signal (same logic as gimlet runbook), print full table, backup to Drive.
**Estimated time:** typically under 1 min.


In [ ]:
# Code cell 5/7 — BASELINE_CHECKPOINT_SIGNAL_STRONG + table print
import csv
import shutil
import statistics
from pathlib import Path

PROGRESS = Path("/content/tqprobe/results/staged/progress.csv")
if not PROGRESS.exists():
    rows = []
    print("⚠️ no progress.csv (run cell 1).")
else:
    try:
        with PROGRESS.open(encoding="utf-8", newline="") as f:
            rows = list(csv.DictReader(f))
    except Exception as e:
        print("⚠️ could not read progress.csv:", e)
        rows = []

ok = [r for r in rows if r.get("status") == "OK"]

if not ok:
    print("BASELINE_CHECKPOINT_SIGNAL_STRONG: NO (no OK rows)")
else:

    def _row_complete(r):
        try:
            return (
                r.get("last_step")
                and r.get("total_steps")
                and int(r["last_step"]) == int(r["total_steps"])
            )
        except (TypeError, ValueError, KeyError):
            return False

    complete = [r for r in ok if _row_complete(r)]
    by_iter = {}
    for r in complete:
        try:
            it = int((r.get("tag") or "").split("_i")[1])
            by_iter.setdefault(it, []).append(float(r["last_loss"]))
        except (IndexError, TypeError, ValueError):
            pass
    trend_ok = (
        100 in by_iter
        and 200 in by_iter
        and 300 in by_iter
        and statistics.median(by_iter[300]) <= statistics.median(by_iter[200]) <= statistics.median(by_iter[100])
    )
    step_avgs = []
    for r in complete:
        if r.get("step_avg_ms"):
            try:
                step_avgs.append(float(r["step_avg_ms"]))
            except ValueError:
                pass
    ckpt_bytes = []
    for r in complete:
        if r.get("ckpt_bytes"):
            try:
                ckpt_bytes.append(int(r["ckpt_bytes"]))
            except ValueError:
                pass
    strong = len(complete) >= 5 and trend_ok

    print(f"successful_rows={len(ok)} complete_rows={len(complete)} trend_ok={trend_ok}")
    if step_avgs:
        print(f"median_step_avg_ms={statistics.median(step_avgs):.1f}")
    if ckpt_bytes:
        print(f"checkpoint_bytes_range=({min(ckpt_bytes)}, {max(ckpt_bytes)})")
    print("BASELINE_CHECKPOINT_SIGNAL_STRONG:", "YES" if strong else "NO")

print("\nFull table:")
for r in rows:
    print(r)

bd = Path("/content/drive/MyDrive/tqprobe_staged")
if bd.parent.exists() and PROGRESS.exists():
    try:
        shutil.copy(PROGRESS, bd / "progress.csv")
    except OSError as e:
        print("⚠️ Drive backup skipped:", e)

---
## Code cell 6: TABLE — formatted CSV
**Goal:** Pretty-print `progress.csv` with `column -t` for a quick visual scan.
**Estimated time:** typically under 1 min.


In [ ]:
%%bash
# Code cell 6/7 — TABLE (column -t progress.csv)
PROG=/content/tqprobe/results/staged/progress.csv
if [ -f "$PROG" ]; then column -s, -t < "$PROG"; else echo '(no progress.csv — run cell 1)'; fi

---
## Code cell 7: MINI_VAL_BPB (TurboQuant staged)
**Goal:** Load the best staged checkpoint (longest run, highest train loss among ties), run a short val slice for `val_bpb`-style metric (same byte accounting as `train_gpt.py`).
**Note:** Uses `forward_logits` + `bigram` args from `Hyperparameters`.


In [ ]:
# CELL 7/7 - MINI_VAL_BPB (TurboQuant staged)
import csv
import importlib.util
import math
import os
from pathlib import Path

import torch
import torch.nn.functional as F
import sentencepiece as spm


def _mini_val():
    backup = Path("/content/drive/MyDrive/tqprobe_staged")
    out = Path("/content/tqprobe/results/staged")
    progress = out / "progress.csv" if (out / "progress.csv").exists() else backup / "progress.csv"

    if not progress.exists():
        print("⏭️ SKIP mini val: no progress.csv (run cell 1 + training cells).")
        return
    if not torch.cuda.is_available():
        print("⏭️ SKIP mini val: no CUDA (enable GPU runtime to run).")
        return
    tg_path = Path("/content/pg/train_gpt.py")
    if not tg_path.is_file():
        print("⏭️ SKIP mini val: train_gpt.py missing (run cell 1).")
        return

    try:
        with progress.open(encoding="utf-8", newline="") as f:
            rows = [r for r in csv.DictReader(f) if r.get("status") == "OK"]
    except Exception as e:
        print("⚠️ could not read progress.csv:", e)
        return

    if not rows:
        print("⏭️ SKIP mini val: no OK rows in progress.csv.")
        return

    def _tag_key(r):
        try:
            return (int((r.get("tag") or "").split("_i")[1]), -float(r["last_loss"]))
        except (IndexError, TypeError, ValueError, KeyError):
            return (0, 0.0)

    best = sorted(rows, key=_tag_key, reverse=True)[0]
    tag = best.get("tag")
    ckpt = out / f"final_model_{tag}.pt"
    if not ckpt.is_file():
        ckpt = backup / f"final_model_{tag}.pt"
    if not ckpt.is_file():
        print(f"⏭️ SKIP mini val: missing checkpoint for {tag}")
        return

    data_path = os.environ.get("DATA_PATH", "/content/openai-pg/data/datasets/fineweb10B_sp1024")
    tokenizer_path = os.environ.get("TOKENIZER_PATH", "/content/openai-pg/data/tokenizers/fineweb_1024_bpe.model")
    vocab_size = int(os.environ.get("VOCAB_SIZE", "1024"))
    train_seq_len = int(os.environ.get("TRAIN_SEQ_LEN", "512"))
    mini_val_seqs = int(os.environ.get("MINI_VAL_SEQS", "64"))

    device = torch.device("cuda")
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    print("tag:", tag, "ckpt:", ckpt)
    print("device:", device, "amp_dtype:", amp_dtype)

    try:
        spec = importlib.util.spec_from_file_location("tg", str(tg_path))
        tg = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(tg)
    except Exception as e:
        print("⚠️ SKIP mini val: could not load train_gpt.py:", e)
        return

    try:
        sp = spm.SentencePieceProcessor(model_file=tokenizer_path)
    except Exception as e:
        print("⚠️ SKIP mini val: tokenizer load failed:", e)
        return

    if int(sp.vocab_size()) != vocab_size:
        print("⚠️ SKIP mini val: VOCAB_SIZE mismatch tokenizer.")
        return

    try:
        val_full = tg.load_validation_tokens(os.path.join(data_path, "fineweb_val_*.bin"), train_seq_len)
    except Exception as e:
        print("⚠️ SKIP mini val: val tokens load failed:", e)
        return

    need = mini_val_seqs * train_seq_len + 1
    if val_full.numel() < need:
        print("⚠️ SKIP mini val: not enough val tokens", val_full.numel(), need)
        return
    val_tokens = val_full[:need].to(dtype=torch.int64)

    base_bytes_lut, has_leading_space_lut, is_boundary_token_lut = tg.build_sentencepiece_luts(
        sp, vocab_size, device
    )

    args = tg.Hyperparameters()
    args.vocab_size = vocab_size
    args.train_seq_len = train_seq_len

    model = tg.GPT(
        vocab_size=args.vocab_size,
        num_layers=args.num_layers,
        model_dim=args.model_dim,
        num_heads=args.num_heads,
        num_kv_heads=args.num_kv_heads,
        mlp_mult=args.mlp_mult,
        tie_embeddings=args.tie_embeddings,
        tied_embed_init_std=args.tied_embed_init_std,
        logit_softcap=args.logit_softcap,
        rope_base=args.rope_base,
        qk_gain_init=args.qk_gain_init,
        bigram_vocab_size=args.bigram_vocab_size,
        bigram_dim=args.bigram_dim,
    ).to(device)
    model = model.bfloat16()
    for m in model.modules():
        if isinstance(m, tg.CastedLinear):
            m.float()
    tg.restore_low_dim_params_to_fp32(model)

    try:
        sd = torch.load(str(ckpt), map_location="cpu")
        model.load_state_dict(sd, strict=True)
    except Exception as e:
        print("⚠️ SKIP mini val: checkpoint load failed:", e)
        return

    model.eval()
    x = val_tokens[:-1].reshape(-1, train_seq_len).to(device)
    y = val_tokens[1:].reshape(-1, train_seq_len).to(device)
    with torch.inference_mode():
        with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=True):
            logits = model.forward_logits(x)
        nll = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)).float(),
            y.reshape(-1),
            reduction="mean",
        ).item()

    prev_ids = x.reshape(-1)
    tgt_ids = y.reshape(-1)
    tb = base_bytes_lut[tgt_ids].to(torch.int16)
    tb += (has_leading_space_lut[tgt_ids] & ~is_boundary_token_lut[prev_ids]).to(torch.int16)
    byte_count = float(tb.sum().item())
    token_count = float(tgt_ids.numel())
    mini_bpb = (nll / math.log(2.0)) * (token_count / byte_count)

    print("\n=== MINI VAL_BPB (TurboQuant staged) ===")
    print("selected_tag:", tag)
    print("checkpoint_bytes:", best.get("ckpt_bytes", ""))
    print("selected_last_loss:", best.get("last_loss", ""))
    print("MINI_VAL_SEQS:", mini_val_seqs, "SEQ_LEN:", train_seq_len)
    print("mini_val_bpb:", f"{mini_bpb:.6f}")


_mini_val()

